In [1]:
import sys
import subprocess
import importlib.util

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyarrow": "pyarrow",
}

for module_name, package_name in required_packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", package_name]
        )

print("Required packages are available.")

Required packages are available.


In [2]:
from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_colwidth", 180)

RANDOM_STATE = 42
LOCAL_DOSES_M = [5.0, 15.0, 30.0]

In [ ]:
from pathlib import Path
import os

# Detect repository root
REPO_ROOT = Path(
    os.environ.get("FORCE_REPO_ROOT", Path.cwd())
).expanduser().resolve()

if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "notebooks").is_dir():
    raise FileNotFoundError(
        "Repository root not found. "
        "Start Jupyter from the repository root or notebooks folder, "
        "or set FORCE_REPO_ROOT."
    )

# Raw dataset directory
DATA_DIR = Path(
    os.environ.get(
        "FORCE_DATA_DIR",
        REPO_ROOT / "data" / "raw"
    )
).expanduser().resolve()

# Output directory
RESULTS_ROOT = Path(
    os.environ.get(
        "FORCE_RESULTS_DIR",
        REPO_ROOT / "outputs"
    )
).expanduser().resolve()

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Data:", DATA_DIR)
print("Outputs:", RESULTS_ROOT)

In [ ]:
PROJECT_ROOT = RESULTS_ROOT

STAGE_01_DIR = (
    PROJECT_ROOT / "01_clean_splits_support"
)

CLEAN_DATA_PATH = (
    STAGE_01_DIR
    / "FORCE_train_cleaned_with_splits.parquet"
)

OLD_REGISTRY_PATH = (
    STAGE_01_DIR
    / "19_final_treatment_registry.csv"
)

OLD_CONTRACT_PATH = (
    STAGE_01_DIR
    / "20_causal_project_contract_v0_2.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT / "01B_fixed_local_windows"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Clean data:", CLEAN_DATA_PATH)
print("Old registry:", OLD_REGISTRY_PATH)
print("Old contract:", OLD_CONTRACT_PATH)
print("Output:", OUTPUT_DIR)

for required_path in [
    CLEAN_DATA_PATH,
    OLD_REGISTRY_PATH,
    OLD_CONTRACT_PATH,
]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)

Clean data : /Users/mohaddesemoghaddam/Desktop/FORCE_Causal_Project/01_clean_splits_support/FORCE_train_cleaned_with_splits.parquet
Old registry: /Users/mohaddesemoghaddam/Desktop/FORCE_Causal_Project/01_clean_splits_support/19_final_treatment_registry.csv
Old contract: /Users/mohaddesemoghaddam/Desktop/FORCE_Causal_Project/01_clean_splits_support/20_causal_project_contract_v0_2.json
Output      : /Users/mohaddesemoghaddam/Desktop/FORCE_Causal_Project/01B_fixed_local_windows


In [4]:
train_clean = pd.read_parquet(
    CLEAN_DATA_PATH
)

old_registry = pd.read_csv(
    OLD_REGISTRY_PATH
)

with open(
    OLD_CONTRACT_PATH,
    "r",
    encoding="utf-8",
) as file:
    old_contract = json.load(file)

required_columns = {
    "WELL",
    "DEPTH_MD",
    "target",
    "GROUP_FOLD",
    "GEO_BLOCK",
}

missing_columns = sorted(
    required_columns
    - set(train_clean.columns)
)

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Rows   :", len(train_clean))
print("Wells  :", train_clean["WELL"].nunique())
print("Classes:", train_clean["target"].nunique())
print("Contract version:", old_contract.get("contract_version"))

Rows   : 1170511
Wells  : 98
Classes: 12
Contract version: 0.2.0


In [5]:
PRIMARY_DIRECT_FEATURES = [
    feature
    for feature in old_contract[
        "primary_direct_treatment_features"
    ]
    if feature in train_clean.columns
]

PATHWAY_FEATURES = [
    feature
    for feature in old_contract[
        "pathway_driver_features"
    ]
    if feature in train_clean.columns
]

AUDIT_FEATURES = list(dict.fromkeys(
    PRIMARY_DIRECT_FEATURES
    + PATHWAY_FEATURES
))

LITHOLOGY_NAME_MAP = {
    30000: "Sandstone",
    65000: "Shale",
    65030: "Sandstone/Shale",
    70000: "Limestone",
    70032: "Chalk",
    74000: "Dolomite",
    80000: "Marl",
    86000: "Anhydrite",
    88000: "Halite",
    90000: "Coal",
    93000: "Basement",
    99000: "Tuff",
}

print("Primary direct features:", PRIMARY_DIRECT_FEATURES)
print("Pathway features:", PATHWAY_FEATURES)
print("Audit features:", AUDIT_FEATURES)

Primary direct features: ['GR', 'NPHI', 'RHOB', 'PEF', 'DTC', 'SP', 'RDEP']
Pathway features: ['CALI']
Audit features: ['GR', 'NPHI', 'RHOB', 'PEF', 'DTC', 'SP', 'RDEP', 'CALI']


In [6]:
well_step_rows = []

for well_name, well_df in train_clean.groupby(
    "WELL",
    sort=True,
):
    depths = (
        well_df["DEPTH_MD"]
        .dropna()
        .sort_values()
        .to_numpy(dtype=float)
    )

    positive_steps = np.diff(depths)
    positive_steps = positive_steps[
        positive_steps > 0
    ]

    well_step_rows.append({
        "WELL": well_name,
        "median_positive_step": (
            float(np.median(positive_steps))
            if len(positive_steps)
            else np.nan
        ),
        "p95_positive_step": (
            float(np.percentile(positive_steps, 95))
            if len(positive_steps)
            else np.nan
        ),
    })

well_depth_steps = pd.DataFrame(
    well_step_rows
)

GLOBAL_MEDIAN_STEP = float(
    well_depth_steps[
        "median_positive_step"
    ].median()
)

well_step_lookup = dict(
    zip(
        well_depth_steps["WELL"],
        well_depth_steps[
            "median_positive_step"
        ],
    )
)

print("Global median depth step:", GLOBAL_MEDIAN_STEP)
display(well_depth_steps.head())

Global median depth step: 0.15200000000004366


,WELL,median_positive_step,p95_positive_step
0,15/9-13,0.152,0.152
1,15/9-15,0.152,0.152
2,15/9-17,0.152,0.152
3,16/1-2,0.152,0.152
4,16/1-6 A,0.152,0.152


In [7]:
def extract_homogeneous_observed_segments(
    dataframe,
    feature,
):
    segment_records = []
    segment_id = 0

    for well_name, well_df in dataframe[
        [
            "WELL",
            "DEPTH_MD",
            "target",
            "GROUP_FOLD",
            "GEO_BLOCK",
            feature,
        ]
    ].groupby(
        "WELL",
        sort=True,
    ):
        well_df = (
            well_df
            .sort_values("DEPTH_MD")
            .reset_index(drop=True)
        )

        depths = well_df[
            "DEPTH_MD"
        ].to_numpy(dtype=float)

        classes = well_df[
            "target"
        ].to_numpy(dtype=int)

        values_observed = (
            well_df[feature]
            .notna()
            .to_numpy(dtype=bool)
        )

        group_fold = int(
            well_df["GROUP_FOLD"].iloc[0]
        )

        geo_block = int(
            well_df["GEO_BLOCK"].iloc[0]
        )

        median_step = well_step_lookup.get(
            well_name,
            GLOBAL_MEDIAN_STEP,
        )

        if (
            not np.isfinite(median_step)
            or median_step <= 0
        ):
            median_step = GLOBAL_MEDIAN_STEP

        gap_threshold = max(
            3.0 * median_step,
            0.5,
        )

        start_index = None

        def close_segment(end_index):
            nonlocal start_index
            nonlocal segment_id

            if start_index is None:
                return

            segment_depths = depths[
                start_index:
                end_index + 1
            ]

            class_code = int(
                classes[start_index]
            )

            segment_id += 1

            segment_records.append({
                "feature": feature,
                "segment_id": segment_id,
                "WELL": well_name,
                "GROUP_FOLD": group_fold,
                "GEO_BLOCK": geo_block,
                "lithology_code": class_code,
                "lithology_name": (
                    LITHOLOGY_NAME_MAP[
                        class_code
                    ]
                ),
                "start_depth": float(
                    segment_depths[0]
                ),
                "end_depth": float(
                    segment_depths[-1]
                ),
                "depth_span_m": float(
                    segment_depths[-1]
                    - segment_depths[0]
                ),
                "row_count": int(
                    len(segment_depths)
                ),
                "median_step_m": float(
                    median_step
                ),
            })

            start_index = None

        for index in range(len(well_df)):
            current_valid = (
                values_observed[index]
                and np.isfinite(depths[index])
            )

            if not current_valid:
                close_segment(index - 1)
                continue

            if start_index is None:
                start_index = index
                continue

            previous_valid = (
                values_observed[index - 1]
                and np.isfinite(depths[index - 1])
            )

            depth_gap = (
                depths[index]
                - depths[index - 1]
            )

            lithology_changed = (
                classes[index]
                != classes[index - 1]
            )

            invalid_gap = (
                not previous_valid
                or not np.isfinite(depth_gap)
                or depth_gap <= 0
                or depth_gap > gap_threshold
            )

            if lithology_changed or invalid_gap:
                close_segment(index - 1)
                start_index = index

        close_segment(
            len(well_df) - 1
        )

    return pd.DataFrame(segment_records)


segment_frames = []

for feature in AUDIT_FEATURES:
    print("Segmenting:", feature)

    feature_segments = (
        extract_homogeneous_observed_segments(
            train_clean,
            feature,
        )
    )

    segment_frames.append(
        feature_segments
    )

homogeneous_segments = pd.concat(
    segment_frames,
    ignore_index=True,
)

print("Total homogeneous segments:", len(homogeneous_segments))
display(homogeneous_segments.head())

Segmenting: GR
Segmenting: NPHI
Segmenting: RHOB
Segmenting: PEF
Segmenting: DTC
Segmenting: SP
Segmenting: RDEP
Segmenting: CALI
Total homogeneous segments: 227872


,feature,segment_id,WELL,GROUP_FOLD,GEO_BLOCK,lithology_code,lithology_name,start_depth,end_depth,depth_span_m,row_count,median_step_m
0,GR,1,15/9-13,1,2,65000,Shale,494.528,509.272,14.744,98,0.152
1,GR,2,15/9-13,1,2,30000,Sandstone,509.424,515.352,5.928,40,0.152
2,GR,3,15/9-13,1,2,65000,Shale,515.504,532.224,16.720,111,0.152
3,GR,4,15/9-13,1,2,30000,Sandstone,532.376,533.896,1.520,11,0.152
4,GR,5,15/9-13,1,2,65000,Shale,534.048,847.168,313.120,2061,0.152


In [8]:
segment_window_records = []

for row in homogeneous_segments.itertuples(
    index=False
):
    for dose_m in LOCAL_DOSES_M:
        required_rows = max(
            2,
            int(
                math.ceil(
                    dose_m
                    / row.median_step_m
                )
            )
            + 1,
        )

        overlapping_starts = max(
            0,
            row.row_count
            - required_rows
            + 1,
        )

        nonoverlap_windows = (
            int(
                math.floor(
                    row.depth_span_m
                    / dose_m
                )
            )
            if row.depth_span_m >= dose_m
            else 0
        )

        segment_window_records.append({
            "feature": row.feature,
            "segment_id": row.segment_id,
            "WELL": row.WELL,
            "GROUP_FOLD": row.GROUP_FOLD,
            "GEO_BLOCK": row.GEO_BLOCK,
            "lithology_code": row.lithology_code,
            "lithology_name": row.lithology_name,
            "dose_m": float(dose_m),
            "required_rows_approx": int(
                required_rows
            ),
            "eligible_overlapping_start_count": int(
                overlapping_starts
            ),
            "eligible_nonoverlap_window_count": int(
                nonoverlap_windows
            ),
        })

segment_window_support = pd.DataFrame(
    segment_window_records
)

display(segment_window_support.head())

,feature,segment_id,WELL,GROUP_FOLD,GEO_BLOCK,lithology_code,lithology_name,dose_m,required_rows_approx,eligible_overlapping_start_count,eligible_nonoverlap_window_count
0,GR,1,15/9-13,1,2,65000,Shale,5.0,34,65,2
1,GR,1,15/9-13,1,2,65000,Shale,15.0,100,0,0
2,GR,1,15/9-13,1,2,65000,Shale,30.0,199,0,0
3,GR,2,15/9-13,1,2,30000,Sandstone,5.0,34,7,1
4,GR,2,15/9-13,1,2,30000,Sandstone,15.0,100,0,0


In [9]:
window_support_by_well = (
    segment_window_support
    .groupby(
        [
            "feature",
            "lithology_code",
            "lithology_name",
            "dose_m",
            "WELL",
            "GROUP_FOLD",
            "GEO_BLOCK",
        ],
        as_index=False,
    )
    .agg(
        eligible_overlapping_start_count=(
            "eligible_overlapping_start_count",
            "sum",
        ),
        eligible_nonoverlap_window_count=(
            "eligible_nonoverlap_window_count",
            "sum",
        ),
        eligible_segment_count=(
            "eligible_nonoverlap_window_count",
            lambda values: int(
                (values > 0).sum()
            ),
        ),
    )
)

window_support_summary = (
    window_support_by_well
    .groupby(
        [
            "feature",
            "lithology_code",
            "lithology_name",
            "dose_m",
        ]
    )
    .agg(
        eligible_overlapping_start_count=(
            "eligible_overlapping_start_count",
            "sum",
        ),
        eligible_nonoverlap_window_count=(
            "eligible_nonoverlap_window_count",
            "sum",
        ),
        eligible_wells=(
            "WELL",
            lambda values: int(
                values[
                    window_support_by_well.loc[
                        values.index,
                        "eligible_nonoverlap_window_count",
                    ] > 0
                ].nunique()
            ),
        ),
        wells_with_at_least_3_windows=(
            "eligible_nonoverlap_window_count",
            lambda values: int(
                (values >= 3).sum()
            ),
        ),
        wells_with_at_least_5_windows=(
            "eligible_nonoverlap_window_count",
            lambda values: int(
                (values >= 5).sum()
            ),
        ),
        grouped_folds_with_windows=(
            "GROUP_FOLD",
            lambda values: int(
                window_support_by_well.loc[
                    values.index
                ]
                .loc[
                    window_support_by_well.loc[
                        values.index,
                        "eligible_nonoverlap_window_count",
                    ] > 0,
                    "GROUP_FOLD",
                ]
                .nunique()
            ),
        ),
        geo_blocks_with_windows=(
            "GEO_BLOCK",
            lambda values: int(
                window_support_by_well.loc[
                    values.index
                ]
                .loc[
                    window_support_by_well.loc[
                        values.index,
                        "eligible_nonoverlap_window_count",
                    ] > 0,
                    "GEO_BLOCK",
                ]
                .nunique()
            ),
        ),
    )
    .reset_index()
)

window_support_summary[
    "local_CATE_support_status"
] = np.select(
    [
        (
            (
                window_support_summary[
                    "eligible_nonoverlap_window_count"
                ] >= 100
            )
            & (
                window_support_summary[
                    "eligible_wells"
                ] >= 10
            )
            & (
                window_support_summary[
                    "wells_with_at_least_3_windows"
                ] >= 5
            )
            & (
                window_support_summary[
                    "grouped_folds_with_windows"
                ] == 5
            )
        ),
        (
            (
                window_support_summary[
                    "eligible_nonoverlap_window_count"
                ] >= 30
            )
            & (
                window_support_summary[
                    "eligible_wells"
                ] >= 5
            )
            & (
                window_support_summary[
                    "grouped_folds_with_windows"
                ] >= 4
            )
        ),
    ],
    [
        "primary_local_CATE",
        "targeted_local_CATE",
    ],
    default="descriptive_or_insufficient",
)

display(
    window_support_summary.loc[
        window_support_summary[
            "feature"
        ].isin(PRIMARY_DIRECT_FEATURES)
    ]
)

,feature,lithology_code,lithology_name,dose_m,eligible_overlapping_start_count,eligible_nonoverlap_window_count,eligible_wells,wells_with_at_least_3_windows,wells_with_at_least_5_windows,grouped_folds_with_windows,geo_blocks_with_windows,local_CATE_support_status
36,DTC,30000,Sandstone,5.0,67691,2722,95,87,83,5,5,primary_local_CATE
37,DTC,30000,Sandstone,15.0,28205,462,72,51,39,5,5,primary_local_CATE
38,DTC,30000,Sandstone,30.0,11644,116,48,17,4,5,5,primary_local_CATE
39,DTC,65000,Shale,5.0,461380,15942,97,96,96,5,5,primary_local_CATE
40,DTC,65000,Shale,15.0,308959,4009,94,92,89,5,5,primary_local_CATE
41,DTC,65000,Shale,30.0,200956,1461,93,74,69,5,5,primary_local_CATE
42,DTC,65030,Sandstone/Shale,5.0,43159,1816,82,67,54,5,5,primary_local_CATE
43,DTC,65030,Sandstone/Shale,15.0,17099,287,43,23,17,5,5,primary_local_CATE
44,DTC,65030,Sandstone/Shale,30.0,7759,67,20,9,4,5,5,targeted_local_CATE
45,DTC,70000,Limestone,5.0,22020,796,46,36,28,5,5,primary_local_CATE


In [ ]:


well_dose_support = (
    window_support_by_well
    .groupby(
        [
            "feature",
            "dose_m",
            "WELL",
            "GROUP_FOLD",
            "GEO_BLOCK",
        ],
        as_index=False,
    )
    .agg(
        eligible_nonoverlap_window_count=(
            "eligible_nonoverlap_window_count",
            "sum",
        ),
        eligible_overlapping_start_count=(
            "eligible_overlapping_start_count",
            "sum",
        ),
    )
)



positive_well_dose_support = (
    well_dose_support.loc[
        well_dose_support[
            "eligible_nonoverlap_window_count"
        ] > 0
    ]
    .copy()
)



global_feature_window_support = (
    positive_well_dose_support
    .groupby(
        [
            "feature",
            "dose_m",
        ],
        as_index=False,
    )
    .agg(
        eligible_nonoverlap_window_count=(
            "eligible_nonoverlap_window_count",
            "sum",
        ),
        eligible_overlapping_start_count=(
            "eligible_overlapping_start_count",
            "sum",
        ),
        eligible_wells=(
            "WELL",
            "nunique",
        ),
        grouped_folds_with_windows=(
            "GROUP_FOLD",
            "nunique",
        ),
        geo_blocks_with_windows=(
            "GEO_BLOCK",
            "nunique",
        ),
    )
)

global_feature_window_support[
    "local_ATE_ready"
] = (
    (
        global_feature_window_support[
            "eligible_nonoverlap_window_count"
        ] >= 500
    )
    & (
        global_feature_window_support[
            "eligible_wells"
        ] >= 30
    )
    & (
        global_feature_window_support[
            "grouped_folds_with_windows"
        ] == 5
    )
)

display(
    global_feature_window_support.sort_values(
        [
            "dose_m",
            "feature",
        ]
    )
)

,feature,dose_m,eligible_nonoverlap_window_count,eligible_overlapping_start_count,eligible_wells,grouped_folds_with_windows,geo_blocks_with_windows,local_ATE_ready
0,CALI,5.0,22614,632553,96,5,5,True
3,DTC,5.0,22638,632425,98,5,5,True
6,GR,5.0,24637,692519,98,5,5,True
9,NPHI,5.0,15246,418251,98,5,5,True
12,PEF,5.0,13744,379671,69,5,5,True
15,RDEP,5.0,24328,682747,98,5,5,True
18,RHOB,5.0,20675,573564,98,5,5,True
21,SP,5.0,18032,505113,68,5,5,True
1,CALI,15.0,5261,393267,96,5,5,True
4,DTC,15.0,5256,391553,98,5,5,True


In [14]:
dose_pivot = (
    window_support_summary
    .pivot_table(
        index=[
            "feature",
            "lithology_code",
            "lithology_name",
        ],
        columns="dose_m",
        values=[
            "eligible_nonoverlap_window_count",
            "eligible_wells",
            "grouped_folds_with_windows",
            "local_CATE_support_status",
        ],
        aggfunc="first",
    )
)

dose_pivot.columns = [
    f"{metric}_{int(dose)}m"
    for metric, dose
    in dose_pivot.columns
]

dose_pivot = (
    dose_pivot
    .reset_index()
)

local_registry = old_registry.merge(
    dose_pivot,
    on=[
        "feature",
        "lithology_code",
        "lithology_name",
    ],
    how="left",
)

status_columns = [
    f"local_CATE_support_status_{int(dose)}m"
    for dose in LOCAL_DOSES_M
]

for column in status_columns:
    if column not in local_registry.columns:
        local_registry[column] = pd.NA

local_registry[
    "final_local_intervention_tier"
] = np.select(
    [
        (
            (
                local_registry[
                    "analysis_tier"
                ] == "primary_confirmatory"
            )
            & local_registry[
                status_columns
            ]
            .eq(
                "primary_local_CATE"
            )
            .all(axis=1)
        ),
        (
            (
                local_registry[
                    "analysis_tier"
                ].isin(
                    [
                        "primary_confirmatory",
                        "targeted_with_caution",
                    ]
                )
            )
            & local_registry[
                status_columns
            ]
            .isin(
                [
                    "primary_local_CATE",
                    "targeted_local_CATE",
                ]
            )
            .any(axis=1)
        ),
    ],
    [
        "confirmatory_all_doses",
        "dose_specific_or_targeted",
    ],
    default="descriptive_only",
)

display(
    local_registry.loc[
        local_registry[
            "feature"
        ].isin(AUDIT_FEATURES)
    ]
)

,feature,lithology_code,lithology_name,observed_rows,supporting_wells,wells_with_at_least_10_rows,wells_with_at_least_25_rows,wells_with_at_least_50_rows,top1_well_share_percent,top3_well_share_percent,top5_well_share_percent,effective_number_of_wells,CATE_support_status,grouped_splits_with_any_rows,grouped_splits_with_at_least_25_rows,grouped_splits_with_at_least_100_rows,grouped_min_rows,grouped_min_wells,grouped_cross_split_status,geo_blocks_with_any_rows,geo_blocks_with_at_least_25_rows,geo_blocks_with_at_least_100_rows,geo_min_rows,geo_min_wells,geographic_cross_split_status,feature_role,analysis_tier,eligible_nonoverlap_window_count_5m,eligible_nonoverlap_window_count_15m,eligible_nonoverlap_window_count_30m,eligible_wells_5m,eligible_wells_15m,eligible_wells_30m,grouped_folds_with_windows_5m,grouped_folds_with_windows_15m,grouped_folds_with_windows_30m,local_CATE_support_status_5m,local_CATE_support_status_15m,local_CATE_support_status_30m,final_local_intervention_tier
12,CALI,88000,Halite,8213,3,3,2,2,52.282966,100.000000,100.000000,2.005092,descriptive_or_insufficient,3,2,2,20,1,limited,2,2,2,3899,1,limited,pathway_driver,descriptive_only,235.0,74.0,33.0,2.0,2.0,2.0,2.0,2.0,2.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
13,CALI,93000,Basement,95,1,1,1,1,100.000000,100.000000,100.000000,1.000000,descriptive_or_insufficient,1,1,0,95,1,limited,1,1,0,95,1,limited,pathway_driver,descriptive_only,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
38,DTC,88000,Halite,8213,3,3,2,2,52.282966,100.000000,100.000000,2.005092,descriptive_or_insufficient,3,2,2,20,1,limited,2,2,2,3899,1,limited,primary_direct,descriptive_only,235.0,74.0,33.0,2.0,2.0,2.0,2.0,2.0,2.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
39,DTC,86000,Anhydrite,1068,5,4,4,4,45.224719,92.509363,100.000000,3.097905,descriptive_or_insufficient,5,4,3,5,1,moderate,3,2,2,5,1,limited,primary_direct,descriptive_only,18.0,4.0,1.0,4.0,2.0,1.0,4.0,2.0,1.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
40,DTC,93000,Basement,93,1,1,1,1,100.000000,100.000000,100.000000,1.000000,descriptive_or_insufficient,1,1,0,93,1,limited,1,1,0,93,1,limited,primary_direct,descriptive_only,2.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
41,GR,88000,Halite,8213,3,3,2,2,52.282966,100.000000,100.000000,2.005092,descriptive_or_insufficient,3,2,2,20,1,limited,2,2,2,3899,1,limited,primary_direct,descriptive_only,235.0,74.0,33.0,2.0,2.0,2.0,2.0,2.0,2.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
42,GR,93000,Basement,103,1,1,1,1,100.000000,100.000000,100.000000,1.000000,descriptive_or_insufficient,1,1,1,103,1,limited,1,1,1,103,1,limited,primary_direct,descriptive_only,3.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
55,NPHI,88000,Halite,4347,3,3,2,2,89.694042,100.000000,100.000000,1.228173,descriptive_or_insufficient,3,2,2,20,1,limited,2,2,2,448,1,limited,primary_direct,descriptive_only,117.0,35.0,14.0,2.0,2.0,2.0,2.0,2.0,2.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
56,NPHI,93000,Basement,103,1,1,1,1,100.000000,100.000000,100.000000,1.000000,descriptive_or_insufficient,1,1,1,103,1,limited,1,1,1,103,1,limited,primary_direct,descriptive_only,3.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0.0,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_or_insufficient,descriptive_only
57,PEF,70032,Chalk,2807,5,5,4,4,35.055219,91.449947,100.000000,3.400154,targeted_with_caution,3,2,2,18,1,limited,2,2,2,222,1,limited,primary_direct,descriptive_only,50.0,8.0,2.0,4.0,2.0,1.0,2

In [ ]:

assert homogeneous_segments[
    "lithology_code"
].notna().all()

primary_30m_support = (
    global_feature_window_support.loc[
        (
            global_feature_window_support[
                "feature"
            ].isin(
                PRIMARY_DIRECT_FEATURES
            )
            & (
                global_feature_window_support[
                    "dose_m"
                ] == 30.0
            )
        )
    ]
)

missing_primary_features = sorted(
    set(PRIMARY_DIRECT_FEATURES)
    - set(
        primary_30m_support[
            "feature"
        ]
    )
)

if missing_primary_features:
    raise RuntimeError(
        "No 30m support record for: "
        f"{missing_primary_features}"
    )

failed_support = primary_30m_support.loc[
    (
        primary_30m_support[
            "eligible_wells"
        ] < 30
    )
    | (
        primary_30m_support[
            "grouped_folds_with_windows"
        ] < 5
    )
]

if len(failed_support):
    raise RuntimeError(
        "Some primary features are not ready "
        "for 30m local ATE:\n"
        f"{failed_support}"
    )

print("All local-window validation gates passed.")

All local-window validation gates passed.


In [16]:
clean_contract = dict(
    old_contract
)

stale_keys = [
    "primary_treatment_features",
    "ATE_only_features",
    "primary_CATE_classes",
    "strict_CATE_classes",
]

for key in stale_keys:
    clean_contract.pop(
        key,
        None,
    )

confirmatory_local_pairs = (
    local_registry.loc[
        local_registry[
            "final_local_intervention_tier"
        ] == "confirmatory_all_doses",
        [
            "feature",
            "lithology_code",
            "lithology_name",
        ],
    ]
    .to_dict(
        orient="records"
    )
)

dose_specific_pairs = (
    local_registry.loc[
        local_registry[
            "final_local_intervention_tier"
        ] == "dose_specific_or_targeted",
        [
            "feature",
            "lithology_code",
            "lithology_name",
        ],
    ]
    .to_dict(
        orient="records"
    )
)

clean_contract.update({
    "contract_version": "0.2.1",
    "local_window_segmentation": {
        "homogeneity_constraint": (
            "feature observed, constant lithology, "
            "continuous depth"
        ),
        "gap_threshold": (
            "max(3 times well median positive depth step, 0.5 m)"
        ),
        "doses_m": LOCAL_DOSES_M,
        "segment_file": str(
            OUTPUT_DIR
            / "01_homogeneous_observed_segments.parquet"
        ),
        "support_registry_file": str(
            OUTPUT_DIR
            / "05_local_intervention_registry.csv"
        ),
    },
    "local_CATE_support_thresholds": {
        "primary_local_CATE": {
            "nonoverlap_windows": 100,
            "eligible_wells": 10,
            "wells_with_at_least_3_windows": 5,
            "grouped_folds_with_windows": 5,
        },
        "targeted_local_CATE": {
            "nonoverlap_windows": 30,
            "eligible_wells": 5,
            "grouped_folds_with_windows": 4,
        },
    },
    "confirmatory_local_feature_class_pairs_all_doses": (
        confirmatory_local_pairs
    ),
    "dose_specific_or_targeted_local_pairs": (
        dose_specific_pairs
    ),
    "rare_class_policy": {
        "Halite": "classification and descriptive intervention analysis only",
        "Anhydrite": "classification and targeted/descriptive analysis only",
        "Basement": "classification only; no cross-well causal claim",
        "Chalk": "targeted CATE with explicit uncertainty and limited geographic transportability",
        "Dolomite": "targeted or confirmatory depending on feature and dose support",
    },
    "baseline_ready": True,
    "next_stage": "02_FORCE_baseline_models_grouped_cv",
})

print(
    json.dumps(
        clean_contract,
        ensure_ascii=False,
        indent=2,
    )[:12000]
)

{
  "contract_version": "0.2.1",
  "random_state": 42,
  "train_file": "/Users/mohaddesemoghaddam/Desktop/train.csv",
  "test_file": "/Users/mohaddesemoghaddam/Desktop/test.csv",
  "claim_scope": "causal effects of controlled measurement interventions on model predictions and errors",
  "excluded_claim": "well logs physically cause geological lithology",
  "primary_causal_path": [
    "TREATMENT",
    "OBSERVED_LOGS",
    "ENGINEERED_FEATURES",
    "MODEL_PREDICTION",
    "PREDICTION_LOSS"
  ],
  "feature_roles": {
    "target": "target",
    "group_identifier": "WELL",
    "depth": "DEPTH_MD",
    "label_confidence": "confidence",
    "spatial_context": [
      "X_LOC",
      "Y_LOC",
      "Z_LOC"
    ],
    "stratigraphic_context": [
      "FORMATION",
      "GROUP"
    ],
    "physical_logs": [
      "GR",
      "NPHI",
      "RHOB",
      "PEF",
      "DTC",
      "SP",
      "CALI",
      "RDEP",
      "RMED",
      "RSHA",
      "RMIC",
      "RXO",
      "DTS",
      "DRHO",
  

In [17]:
well_depth_steps.to_csv(
    OUTPUT_DIR
    / "00_well_depth_steps.csv",
    index=False,
)

homogeneous_segments.to_parquet(
    OUTPUT_DIR
    / "01_homogeneous_observed_segments.parquet",
    index=False,
)

segment_window_support.to_parquet(
    OUTPUT_DIR
    / "02_segment_window_support.parquet",
    index=False,
)

window_support_by_well.to_csv(
    OUTPUT_DIR
    / "03_window_support_by_feature_class_well_dose.csv",
    index=False,
)

window_support_summary.to_csv(
    OUTPUT_DIR
    / "04_window_support_summary.csv",
    index=False,
)

local_registry.to_csv(
    OUTPUT_DIR
    / "05_local_intervention_registry.csv",
    index=False,
)

global_feature_window_support.to_csv(
    OUTPUT_DIR
    / "06_global_feature_window_support.csv",
    index=False,
)

with open(
    OUTPUT_DIR
    / "07_causal_project_contract_v0_2_1.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        clean_contract,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved outputs in:")
print(OUTPUT_DIR)

Saved outputs in:
/Users/mohaddesemoghaddam/Desktop/FORCE_Causal_Project/01B_fixed_local_windows
